In [ ]:
# 验证新版 viz：① 不读库、手写三个正方形就能画 ② 小样本抑制 ③ 地图颜色可设、可还原 ④ 旧接口还在
import numpy as np
import matplotlib.pyplot as plt
from signal_insight import viz

sq = np.array([[0, 0], [1, 0], [1, 1], [0, 1]]) * 0.05
toy = viz.maps.Regions({"甲": sq + [123.30, 41.8], "乙": sq + [123.35, 41.8], "丙": sq + [123.40, 41.8]})
v = {"甲": 3, "乙": 120, "丙": 2}                      # 甲、丙不足 5 人

fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
toy.base(ax)                                           # 底图：只由边界决定
pc = toy.fill(ax, v, min_value=5)                      # 人数要抑制：不足 5 的区画斜线
rt = toy.text(ax, v, min_value=5)                      # 同一个门槛：数字写 "<5"
print(toy)
print("被藏的区：", pc.suppressed)
print("区内数字：", sorted(t.get_text() for t in rt.nums.values()))

before = viz.maps.colors.hot
with viz.maps.colors.use(hot="#D64545"):               # 临时改地图的强调色
    inside = viz.maps.colors.hot
print("地图强调色：", before, "→", inside, "→", viz.maps.colors.hot)
print("旧接口：", viz.maps.DistrictMap.__name__, "；热力矩阵：", viz.matrix.__name__)
plt.close(fig)


In [ ]:
%load_ext autoreload
%autoreload 2
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT
viz.apply_theme()                          # 不写主题名 = config.THEME（默认 default）
con = sqlite3.connect(config.DB_PATH)
DAY = config.ANALYSIS_DAY
NB = "D04_这份数据是谁"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}；制图：{config.AUTHOR}"
pd.set_option("display.max_columns", 20)

In [ ]:
# 今天要用的两组多边形：读库在这里（调用方），还原交给 geo
DISTRICTS = geo.rings_from_points(pd.read_sql("SELECT * FROM ods_district_boundary", con), "district")
AOI = geo.rings_from_points(pd.read_sql("SELECT * FROM ods_aoi_v2", con), "name")
# 全市区县底图：外框、长宽比、每个区的落点只由边界决定，算一次，今天的区县图都用它。
# 不叫 geo：geo 已经是模块名，覆盖掉后面就调不了 geo.haversine
city = viz.maps.Regions(DISTRICTS)
print(city)
print(f"区县 {len(DISTRICTS)} 个，商圈 {len(AOI)} 个")

In [ ]:
# 把「清洗后的信令」和「扇区维度表」拼成带位置的宽表 sig，今天所有的分析都从它出发。
dwd = pd.read_sql("SELECT imsi, ts, lac_cell FROM dwd_signal", con, parse_dates=["ts"])
dim = pd.read_sql("SELECT * FROM dim_cell", con)
# inner 而不是 left：D03 已经确认关联率 99.998%，关联不上的 8 条没有坐标，留着只会在每个空间统计里变成空值。
# 先验证关联率、再决定用 inner，顺序不能反。
sig = dwd.merge(dim, on="lac_cell", how="inner")
n_users = sig["imsi"].nunique()
print(f"带位置的信令 {len(sig):,} 条，用户 {n_users:,} 个")

In [ ]:
# 每个区县当天有多少比例的样本用户到过。一个人一天能到好几个区，所以占比加起来会超过 100%。
reach = (sig.groupby("district")["imsi"].nunique() / n_users).rename("到访用户占比")
reach = reach.drop("市外", errors="ignore").sort_values().reset_index()
reach.to_sql("ads_district_reach", con, if_exists="replace", index=False)
reach.tail(5)

In [ ]:
# ── 出图 D4_01：区县地图 + 条形图 ──
# 地图回答"集中在哪"，条形回答"具体差多少"。版面由这里的 plt.subplots 决定，地图只往给它的 ax 上加层。
r = (pd.read_sql("SELECT * FROM ads_district_reach", con)
       .set_index("district")["到访用户占比"].sort_values(ascending=False))   # 大 → 小，条形从上往下
FAR = ["新民市", "辽中区", "法库县", "康平县"]       # 远郊四区县
assert set(r.index[-4:]) == set(FAR)                 # 它们正好垫底；数据变了这里会先报错
top_name, top_v = r.index[0], r.iloc[0]
bot_name, bot_v = r.index[-1], r.iloc[-1]
ratio = top_v / bot_v                                # 最高 ÷ 最低
cut = r.index.get_loc(FAR[0])                        # 远郊段从第几行开始
jump = r.iloc[cut - 1] / r.iloc[cut]                 # 断档处掉了几倍


def pct(v: float) -> str:
    """百分比（v 是 0~1 的比例，规范第 6 节）：一般取整，小于 10% 留一位，小于 1% 留两位——很小的值不写成 0%。"""
    p = v * 100
    return f"{p:.0f}%" if p >= 10 else f"{p:.1f}%" if p >= 1 else f"{p:.2f}%"


fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(12, 6.4), width_ratios=[6 * city.aspect + 3.2, 5.6],
                                     layout="constrained")
# 左图：底图 → 填色 → 描边 → 引线标注，一层一层往上叠
# 占比是比例、不是人数，不用小样本抑制；跨两个数量级（0.3% → 97%），用对数色阶
city.base(ax_map, scale=False, north=dict(x=0.93, y=0.95))
city.fill(ax_map, r, norm="log")
city.edges(ax_map, FAR, color=MAIN, lw=0.9)          # 远郊四区县整圈描蓝，和右图对上
city.label(ax_map, r, fmt=pct, hot=FAR, split="balance")

# 右图：横向条形，远郊四区县用主色
y = np.arange(len(r))
ax_bar.barh(y, r.values * 100, height=0.62, color=[MAIN if n in FAR else GRAY for n in r.index])
for i, (name, v) in enumerate(r.items()):
    ax_bar.text(v * 100 + 1.2, i, pct(v), va="center", fontsize=10, color=MAIN if name in FAR else INK,
                fontweight="bold" if name in FAR else "normal")
ax_bar.set_yticks(y, r.index, fontsize=10.5)
ax_bar.set_ylim(len(r) - 0.4, -1.4)                  # 顶上留一行放列头
ax_bar.set_xlim(0, 118)                              # 条形从 0 开始，右边留出数值的位置
ax_bar.set_xticks([]); ax_bar.grid(False); ax_bar.tick_params(length=0)
for sp in ax_bar.spines.values():
    sp.set_visible(False)
ax_bar.text(0, -1.3, "到访用户占比（分母 = 全部用户）", fontsize=10.5, color=INK, va="center")
ax_bar.axhline(cut - 0.5, color=MUTED, lw=0.8, ls=(0, (3, 3)), xmin=0.02, xmax=0.98)   # 断档线
ax_bar.text(118, cut - 0.62, f"此处降到 1/{jump:.1f}", ha="right", va="bottom", fontsize=9.5, color=MAIN)

title = f"{top_name}被 {pct(top_v)} 的用户到访，{bot_name}只有 {pct(bot_v)}，相差 {ratio:.0f} 倍"
viz.finish(fig, title,
           "左图填色越深到访用户越多（对数色阶），区名后为到访用户占比；右图按占比从大到小排序；蓝色 = 远郊四区县\n"
           "一个用户可出现在多个区，各区分别计数，合计远超 100%，不能相加；口径：dwd_signal × dim_cell，市外站点不计入",
           SOURCE, "D4_01_各区到访用户占比.png")
plt.show()

In [ ]:
site_users = (sig.groupby(["site_id", "lng", "lat"])["imsi"].nunique()
                 .rename("用户数").reset_index()
                 .sort_values("用户数", ascending=False, ignore_index=True))
site_users["用户占比"] = site_users["用户数"] / n_users
site_users.head(5)


In [ ]:
# 热点中心：前 5 个站点按用户数加权平均。一座基站覆盖几百米，第一名站点的坐标只是铁塔的位置，不是人群的中心。
# 这个坐标算一次就写进 dim_hotspot，不随日期重算：否则每天的"西塔"都是一个略微不同的地方，跨日对比失去意义。
top5 = site_users.head(5)
hot_lng = np.average(top5["lng"], weights=top5["用户数"])
hot_lat = np.average(top5["lat"], weights=top5["用户数"])
poi = pd.read_sql("SELECT * FROM ods_poi_purpose", con)
poi["距热点(米)"] = geo.haversine(poi["lat"], poi["lng"], hot_lat, hot_lng).round()
poi.nsmallest(6, "距热点(米)")[["name", "type", "距热点(米)"]]

In [ ]:
dim_hotspot = pd.DataFrame([{"name": "西塔", "lng": hot_lng, "lat": hot_lat, "radius_m": 500,
                             "依据": "连接用户数前 5 的站点按用户数加权的中心；附近为韩百商场、鑫韩城一带"}])
dim_hotspot.to_sql("dim_hotspot", con, if_exists="replace", index=False)
dim_hotspot


In [ ]:
# ── 出图 D4_02：左图内五区六边形热力 → 右图放大到西塔，基站气泡 + 500 米圈 ──
# 全市视野下 2 公里的方框只有一个点大，所以左图只画内五区；版面（两张子图多宽）由这里决定。
CORE = ["铁西区", "和平区", "沈河区", "皇姑区", "大东区"]
hs = dim_hotspot.iloc[0]
core = viz.maps.Regions({n: DISTRICTS[n] for n in CORE})
fig, (ax_map, ax_z) = plt.subplots(1, 2, figsize=(12, 6.9), width_ratios=[core.aspect * 1.08, 1],
                                   layout="constrained")

# 左图：六边形热力（规范第 4 节：空间分布不画几千个散点）。
# 口径 nunique：格内连过任一基站的人去重计数，一人只算一次；不足 5 人的格子不画（规范第 7 节，默认就开着）
core.base(ax_map, north=dict(x=0.95, y=0.95))
hb = core.hexbin(ax_map, sig, 600, value="imsi", how="nunique")   # 格宽 600 米，写进色条
core.text(ax_map, dict.fromkeys(CORE, np.nan), alias={"沈河区": "沈河"},  # 只写区名，给读者定位
          overflow="hide")                                               # 沈河区太窄，写简称；还放不下就不写
viz.maps.colorbar(hb, ax_map, label=f"格内到过的用户数（去重，人；格宽 {hb.size_m:,.0f} 米）")

# 右图：先画底色和区界（不撑满），再 zoom：左图画放大框，右图设成框里的范围、同色边框
core.base(ax_z, fill=False, outline=False, north=False, borders=dict(color=GRAY, lw=0.9, ls=(0, (4, 3))))
viz.maps.zoom(ax_map, ax_z, hs["lng"], hs["lat"], 1_000, label=hs["name"])

# 基站气泡：面积 ∝ 人数；前 5 名用主色；不足 5 人的站点不画（默认 min_value=5）
top5_ids = site_users["site_id"].head(5)
bb = core.bubbles(ax_z, site_users, "用户数", hot=site_users["site_id"].isin(top5_ids),
                  vmax=site_users["用户数"].max(), title="气泡面积 = 连接用户数（人）")
viz.maps.circle(ax_z, hs["lng"], hs["lat"], hs["radius_m"])                 # 按米画圆，画出来是正圆
ry = viz.maps.m2deg(hs["radius_m"], hs["lat"])[1]
ax_z.text(hs["lng"], hs["lat"] + ry * 1.04, f"{hs['name']} {hs['radius_m']:,.0f} 米", ha="center", va="bottom",
          fontsize=10, color=MAIN, fontweight="bold", path_effects=viz.maps.HALO, zorder=6)

top1 = site_users.iloc[0]
title = f"连接人数最多的站点集中在西塔，排第一的站点就有 {top1['用户占比']:.0%} 的用户"
viz.finish(fig, title,
           f"左图：内五区六边形热力，格内到过的用户去重计数（对数色阶），不足 5 人的格子不画；方框 = 右图范围\n"
           f"右图：西塔周边 2×2 公里，气泡 = 基站，面积与连接用户数成正比，蓝色 = 前 5 名；虚线圈 = 热点中心 500 米",
           SOURCE, "D4_02_西塔热点.png")
plt.show()

In [ ]:
def draw_d4_02(sig, districts, dim_hotspot, site_users, poi, source, save=False):
    core_names = ["铁西区", "和平区", "沈河区", "皇姑区", "大东区"]
    core = viz.maps.Regions({n: districts[n] for n in core_names})
    hs = dim_hotspot.loc[dim_hotspot["name"].eq("西塔")].iloc[0]
    lng, lat, radius_m = float(hs["lng"]), float(hs["lat"]), float(hs["radius_m"])
    sites = site_users.sort_values("用户数", ascending=False).copy()
    top = sites.head(5)
    distances = geo.haversine(sig["lat"], sig["lng"], lat, lng)
    reach_n = sig.loc[distances <= radius_m, "imsi"].nunique()
    reach_pct = reach_n / sig["imsi"].nunique()
    top_in = int((geo.haversine(top["lat"], top["lng"], lat, lng) <= radius_m).sum())

    fig, (amap, az) = plt.subplots(1, 2, figsize=(13, 7.8), width_ratios=[core.aspect * 1.06, 1],
                                  layout="constrained")
    core.base(amap, north=dict(x=0.94, y=0.94))
    hb = core.hexbin(amap, sig, 900, value="imsi", how="nunique", min_value=5)
    core.text(amap, dict.fromkeys(core_names, np.nan), alias={"沈河区": "沈河"}, overflow="hide")
    viz.maps.colorbar(hb, amap, label="格内到过的用户（去重，人；格宽 900 米）")
    amap.set_title("A  内五区：西塔在城市中的位置", loc="left", fontsize=11, pad=12)

    core.base(az, fill=False, outline=False, north=False,
              borders=dict(color="#C4CDD8", lw=0.8, ls=(0, (4, 3))))
    viz.maps.zoom(amap, az, lng, lat, 1_000, label="西塔", connect=True)
    core.bubbles(az, sites, "用户数", hot=sites["site_id"].isin(top["site_id"]), min_value=5,
                 color="#C8CFD9", hot_color=MAIN, smax=1550, alpha=0.8,
                 vmax=sites["用户数"].max(), ref=[500, 1000, 2000],
                 title="气泡面积 = 连接用户数（人）")
    viz.maps.circle(az, lng, lat, radius_m)
    az.scatter([lng], [lat], marker="+", s=120, color=INK, lw=1.7, zorder=7)
    ry = viz.maps.m2deg(radius_m, lat)[1]
    az.text(lng, lat + ry * 1.08, f"西塔 {radius_m:,.0f} 米", ha="center", fontsize=11,
            color=MAIN, fontweight="bold", path_effects=viz.maps.HALO, zorder=7)
    offsets = [(-35, 40), (40, 32), (-48, -32), (44, -35), (-65, 0)]
    for (_, row), (dx, dy) in zip(top.iterrows(), offsets):
        if row["用户数"] < 5:
            continue
        az.annotate(f"{row['用户数']:,.0f} 人", (row["lng"], row["lat"]),
                    xytext=(dx, dy), textcoords="offset points", ha="right" if dx < 0 else "left",
                    va="center", fontsize=10, color=MAIN, fontweight="bold",
                    path_effects=viz.maps.HALO, zorder=8,
                    arrowprops=dict(arrowstyle="-", color=MAIN_MID, lw=0.8))
    if poi is not None and len(poi):
        p = poi.copy()
        p["_distance"] = geo.haversine(p["lat"], p["lng"], lat, lng)
        nearest = p.loc[p["_distance"].idxmin()]
        az.annotate(f"{nearest['name']}\n距加权中心 {nearest['_distance']:.0f} 米", (lng, lat),
                    xytext=(0.06, 0.12), textcoords="axes fraction", fontsize=9.5, color=INK,
                    path_effects=viz.maps.HALO, zorder=9,
                    arrowprops=dict(arrowstyle="-", color=INK, lw=0.8))
    az.set_title("B  西塔周边 2×2 公里｜站点、中心、地标", loc="left", fontsize=11, pad=12)
    top_note = "前五站点均在圈内" if top_in == 5 else f"前五站点有 {top_in} 个在圈内"
    title = f"西塔 {radius_m:,.0f} 米圈触达 {reach_pct:.0%} 用户，{top_note}"
    finished = _finish_preview(fig, title,
        f"左图为格内用户去重热力；右图为 2×2 公里放大，蓝色为前 5 站点，+ 为加权中心；圈内触达 {reach_n:,} 人\n"
        "气泡面积为站点连接人数，站点间有重复、不能相加；不足 5 人不画；触达按圈内任一站点去重；口径：dwd_signal × dim_cell",
        source, 'D4_02_西塔热点.png', save)
    return finished, hb



fig_d4_02, hb = draw_d4_02(sig, DISTRICTS, dim_hotspot, site_users, poi, SOURCE, save=SAVE)
plt.show()


In [ ]:
# 格子统计在 hb.cells 里：最热的格子在哪、多少人
hb.cells.head(3)[["lng", "lat", "value", "n_rows"]]

In [ ]:
# 每个人当天离西塔最近有多近；不同半径下有多少人算"到过"。半径一换，人数就变（D8 的预演）。
sig["到西塔(米)"] = geo.haversine(sig["lat"], sig["lng"], hot_lat, hot_lng)
user_min = sig.groupby("imsi")["到西塔(米)"].min()
radius = pd.DataFrame({"半径(米)": [300, 500, 1000, 2000, 5000]})
radius["到过的用户占比"] = [(user_min <= r_).mean() for r_ in radius["半径(米)"]]
radius.to_sql("ads_hotspot_radius", con, if_exists="replace", index=False)
radius

In [ ]:
# ── 出图 D4_03：ECDF ──
# 读者想问"多少人在 500 米以内"，这是累计问题：ECDF 直接读纵轴，直方图要靠眼睛把柱子加起来。
p500 = (user_min <= 500).mean()
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.ecdf(user_min.clip(lower=10), color=MAIN, lw=2.2)
ax.set_xscale("log")                       # 距离跨三个数量级
ax.axvline(500, color=MUTED, lw=1, ls=(0, (4, 3)))
ax.annotate(f"500 米以内：{p500:.0%}", (500, p500), xytext=(10, -18), textcoords="offset points",
            fontsize=10.5, color=INK, fontweight="bold")
ax.scatter([500], [p500], color=MAIN, zorder=3)
ax.set_xticks([100, 300, 1000, 3000, 10_000, 30_000], ["100 米", "300 米", "1 公里", "3 公里", "10 公里", "30 公里"])
ax.yaxis.set_major_formatter("{x:.0%}")
ax.set_xlabel("当天离西塔最近的距离（对数刻度）", color=MUTED)
ax.grid(axis="x", visible=False)
viz.finish(fig, f"{p500:.0%} 的用户当天到过西塔 500 米以内",
           "曲线上一点 = 当天离西塔最近距离不超过横轴数值的用户占比；距离按所连站点的坐标计算；口径：dwd_signal × dim_cell",
           SOURCE, "D4_03_离西塔最近距离.png")
plt.show()

In [ ]:
# 两条曲线：每小时有多少人在西塔（存量），每个人第一次出现在西塔的时刻（流量）。D1 留的"23 点峰值"靠它揭晓。
near = sig[sig["到西塔(米)"] <= 500]
hourly_near = near.groupby(near["ts"].dt.hour)["imsi"].nunique().reindex(range(24), fill_value=0)
first_hour = near.groupby("imsi")["ts"].min().dt.hour.value_counts(normalize=True).reindex(range(24), fill_value=0)
xita_hourly = pd.DataFrame({"小时": range(24), "在场人数": hourly_near.values, "首次出现占比": first_hour.values})
xita_hourly.to_sql("ads_xita_hourly", con, if_exists="replace", index=False)
xita_hourly.iloc[14:21]


In [ ]:
# ── 出图 D4_04：上下两个面板（不用双 Y 轴）──
# 上面是"人"（几千），下面是"占比"（0–40%）；双轴的交点和高低完全由刻度决定，所以分成两个面板、共用横轴。
x = pd.read_sql("SELECT * FROM ads_xita_hourly", con)
pk = x.loc[x["在场人数"].idxmax()]
p17_19 = x.loc[x["小时"].between(17, 19), "首次出现占比"].sum()
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5.6), layout="constrained", sharex=True)
for ax_ in (a1, a2):
    ax_.axvspan(16.5, 19.5, color=MAIN_LIGHT, lw=0, zorder=0)
a1.plot(x["小时"], x["在场人数"], color=MAIN, lw=2.2, marker="o", ms=3.5)
a1.text(pk["小时"] - 0.5, pk["在场人数"] * 1.07, f"{int(pk['小时'])} 点 {int(pk['在场人数']):,} 人",
        ha="right", va="center", fontsize=9.5, color=INK, fontweight="bold")
a1.set_ylim(0, x["在场人数"].max() * 1.15)
a1.yaxis.set_major_formatter("{x:,.0f}")
a1.set_title("西塔 500 米内每小时在场人数（人）", loc="left", fontsize=10, color=INK, fontweight="bold")
a2.plot(x["小时"], x["首次出现占比"], color=GRAY_DARK, lw=2, marker="o", ms=3.5)
e_ = x[x["小时"].between(17, 19)]
a2.plot(e_["小时"], e_["首次出现占比"], color=MAIN, lw=2.6, marker="o", ms=5)
a2.text(18, e_["首次出现占比"].max() + 0.015, f"17–19 点合计 {p17_19:.0%}", ha="center", fontsize=9.5,
        color=MAIN, fontweight="bold")
a2.text(0.3, x["首次出现占比"].iloc[0] + 0.012, f"0 点已在西塔 {x['首次出现占比'].iloc[0]:.0%}（前一晚没走）",
        fontsize=9, color=MUTED)
a2.set_ylim(0, x["首次出现占比"].max() * 1.3)
a2.yaxis.set_major_formatter("{x:.0%}")
a2.set_title("当天第一次出现在西塔的时刻（占到过西塔的人）", loc="left", fontsize=10, color=INK, fontweight="bold")
a2.set_xticks(range(0, 24, 3), [f"{h} 点" for h in range(0, 24, 3)]); a2.set_xlim(-0.5, 23.5)
title = f"{p17_19:.0%} 的人第一次出现在西塔是 17–19 点，在场人数随之在 {int(pk['小时'])} 点升到 {int(pk['在场人数']):,}"
viz.finish(fig, title, "到过西塔 500 米内的用户；上下两个面板共用横轴；浅蓝底 = 17–19 点；口径：dwd_signal × dim_cell",
           SOURCE, "D4_04_西塔在场与首次出现.png")
plt.show()

In [ ]:
# D1 的疑点：为什么 23 点信令最多？看各小时的信令有多少发生在西塔 1 公里内
share_near = sig.assign(小时=sig["ts"].dt.hour, 近西塔=sig["到西塔(米)"] <= 1000).groupby("小时")["近西塔"].mean()
print(f"10 点的信令 {share_near[10]:.0%} 发生在西塔 1 公里内；23 点是 {share_near[23]:.0%}")


In [ ]:
# IMSI = 3 位国家码 + 2 位运营商码 + 用户编号，str[3:5] 就是运营商
mnc = sig.drop_duplicates("imsi")["imsi"].str[3:5].value_counts().rename_axis("运营商代码").reset_index(name="用户数")
mnc["含义"] = mnc["运营商代码"].map({"00": "中国移动", "02": "中国移动", "07": "中国移动"}).fillna("其他")
mnc


In [ ]:
quality.dq_record(con, "DQ-15", "代表性", "样本圈选口径", "dwd_signal",
                  f"{top_v:.0%} 的用户到过和平区，{p500:.0%} 到过西塔 500 米内", "警告",
                  evidence="热点位于西塔；在场人数 17 点后猛增；首次出现时间 16→17 点跳升过陡",
                  decision="结论只针对“10 月 3 日傍晚后到访西塔的人群”；圈选条件需向数据提供方确认")
quality.dq_record(con, "DQ-16", "代表性", "运营商覆盖", "dwd_signal.imsi", "全部为中国移动用户", "警告",
                  evidence="IMSI 运营商代码只有 00、02、07", decision="人数只代表移动用户；推算总人数需按移动市场份额扩样")


In [ ]:
# 每人每小时一个位置（该小时信令位置的中位数：平均会落到几个站点之间的空地上），再归到五类区域
uh = sig.assign(hour=sig["ts"].dt.hour).groupby(["imsi", "hour"], as_index=False)[["lng", "lat"]].median()
uh["district"] = pd.Series(geo.point_in_polygons(uh["lng"], uh["lat"], DISTRICTS)).fillna("市外").to_numpy()
uh["aoi"] = geo.point_in_polygons(uh["lng"], uh["lat"], AOI)
uh["dist_xita_m"] = geo.haversine(uh["lat"], uh["lng"], hot_lat, hot_lng)
CITY = ["沈河区", "皇姑区", "铁西区", "大东区", "浑南区", "于洪区", "沈北新区", "苏家屯区"]
order = ["西塔 500 米内", "太原街、中街商圈", "和平区其他", "其他城区", "远郊县市及市外"]
uh["区域"] = np.select([uh["dist_xita_m"] <= 500, uh["aoi"].isin(["太原街商圈", "中街商圈"]),
                        uh["district"] == "和平区", uh["district"].isin(CITY)], order[:4], default=order[4])
cnt = uh.groupby(["hour", "区域"]).size().unstack(fill_value=0).reindex(index=range(24), columns=order, fill_value=0)
share = cnt.div(cnt.sum(axis=1), axis=0) * 100
share.round(1).loc[[4, 10, 15, 19, 23]]


In [ ]:
# 对账
day_out = (share.loc[10:16, "其他城区"] + share.loc[10:16, "远郊县市及市外"]).mean()
xs = share["西塔 500 米内"]
print(f"✅ 对账通过：{len(uh):,} 个“人 × 小时”；白天 10–16 点 {day_out:.1f}% 在和平区以外；西塔占比 19 点最高（{xs.max():.1f}%）")


In [ ]:
# ── 出图 D4_05：百分比堆叠面积图 ──
# 五类区域互斥且穷尽（每人每小时只落进一类，合计 100%），所以可以堆叠；D4_01 就不能堆（合计超过 100%）。
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.stackplot(share.index, share.T.values, colors=[MAIN, MAIN_MID, "#E4E7EB", GRAY, GRAY_DARK],
             edgecolor="white", linewidth=0.8)
ax.set_xlim(0, 23); ax.set_ylim(0, 100)
ax.set_xticks(range(0, 24, 3), [f"{h} 点" for h in range(0, 24, 3)])
ax.set_yticks(range(0, 101, 25), [f"{v}%" for v in range(0, 101, 25)])
ax.grid(False)
mids = (share.loc[23].cumsum() - share.loc[23] / 2).to_numpy()   # 右侧直接标区域名，挤在一起的往上推开
ys = mids.copy()
for i in range(1, len(ys)):
    ys[i] = max(ys[i], ys[i - 1] + 7)
for g_, ym, y_ in zip(order, mids, ys):
    ax.annotate(g_, xy=(23, ym), xytext=(23.6, min(y_, 97)), va="center", fontsize=10, annotation_clip=False,
                color=MAIN if g_.startswith("西塔") else INK, fontweight="bold" if g_.startswith("西塔") else "normal",
                arrowprops=dict(arrowstyle="-", color=FAINT, lw=0.6, shrinkA=0, shrinkB=0))
peak_h = int(xs.idxmax())
ax.text(peak_h, xs[peak_h] / 2, f"{peak_h} 点\n{xs[peak_h]:.0f}%", ha="center", va="center", color="white",
        fontsize=10, fontweight="bold")
c12 = share.loc[12].cumsum()
ax.text(12, c12["和平区其他"] + share.loc[12, "其他城区"] / 2, f"白天 {day_out:.0f}% 在和平区以外",
        ha="center", va="center", color=INK, fontsize=11)
title = f"白天 {day_out:.0f}% 的人在和平区以外，{peak_h} 点有 {xs[peak_h]:.0f}% 聚到西塔：样本圈的是人，不是地方"
viz.finish(fig, title,
           "每小时有信令的样本用户在各区域的占比；位置 = 该小时信令位置的中位数；西塔 = 热点 500 米内，商圈 = v2 边界\n"
           "D9 会用停留点修正位置，数字略有变化；口径：dwd_signal × dim_cell",
           SOURCE, "D4_05_城市潮汐.png")
plt.show()

In [ ]:
# ── 出图 D4_06：同一张全市底图画 5 遍（小多图），每张一个时刻，看人在哪些区 ──
# 规范第 5 节：前后对比拆成小多图，共用一个色阶，颜色才能互相比。
# 格子里是人数，要小样本抑制：min_value=5，不足 5 人的区画斜线；色阶也从 5 起（LogNorm 的下限）
from matplotlib.colors import LogNorm

HOURS = [4, 10, 15, 19, 23]
by_d = (uh[uh["district"] != "市外"].groupby(["district", "hour"]).size()
          .unstack(fill_value=0).reindex(columns=HOURS, fill_value=0))
by_d.columns = [f"{h} 点" for h in HOURS]
fig, axes, cb = city.facets(by_d, ncols=5, height=2.6, norm=LogNorm(5, by_d.to_numpy().max()),
                            cmap=viz.maps.heat_cmap(), hot_max=False, min_value=5,
                            label="该小时在这个区的用户数（人，对数色阶）")
hp, rest = by_d.loc["和平区"], by_d.drop(index="和平区").sum()
title = (f"和平区从 10 点的 {hp['10 点']:,} 人涨到 23 点的 {hp['23 点']:,} 人，"
         f"其余区县同期从 {rest['10 点']:,} 降到 {rest['23 点']:,}")
viz.finish(fig, title,
           "每张图一个时刻，颜色 = 该小时位置落在这个区的用户数（每人每小时算一次），五张图同一色阶；"
           "斜线 = 不足 5 人（不公布）\n位置 = 该小时信令位置的中位数；市外不计；口径：dwd_signal × dim_cell",
           SOURCE, "D4_06_各时刻人在哪些区.png")
plt.show()

In [ ]:
near = sig[sig["到西塔(米)"] <= 500]
hourly_near = near.groupby(near["ts"].dt.hour)["imsi"].nunique().reindex(range(24), fill_value=0)
first_n = near.groupby("imsi")["ts"].min().dt.hour.value_counts().reindex(range(24), fill_value=0)
xita_hourly = pd.DataFrame({"小时": range(24), "在场人数": hourly_near.values,
                            "首次出现人数": first_n.values, "首次出现占比": (first_n / first_n.sum()).values})
xita_hourly.to_sql("ads_xita_hourly", con, if_exists="replace", index=False)

In [ ]:
# ── 出图 D4_04：在场人数拆成"这一小时第一次来"和"之前来过"，叠一条累计线（同单位，同一根轴）──
x = pd.read_sql("SELECT * FROM ads_xita_hourly", con).set_index("小时")
new = x["首次出现人数"]
old = x["在场人数"] - new                      # 之前出现过、这一小时又在西塔的人
cum = new.cumsum()                             # 截至这一小时累计到过西塔的人
V = int(cum.iloc[-1])
p17_19 = new.loc[17:19].sum() / V
jump = new[17] / new[16]
pk = int(x["在场人数"].idxmax())
h = x.index.to_numpy()

fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
ax.axvspan(16.5, 19.5, color=MAIN_LIGHT, lw=0, zorder=0)
ax.bar(h, new, width=0.72, color=[GRAY_DARK if v == 0 else MAIN for v in h], zorder=2, label="这一小时第一次出现")
ax.bar(h, old, bottom=new, width=0.72, color=GRAY, zorder=2, label="之前出现过、这一小时又在")
ax.plot(h, cum, color=INK, lw=1.6, marker="o", ms=3.2, zorder=3, label="截至该小时累计到过（去重）")
ax.legend(loc="upper left", fontsize=9.5, handlelength=1.4)
ax.text(23.5, cum[23], f" 累计 {V:,} 人", va="center", fontsize=10, color=INK, fontweight="bold")
ax.annotate(f"16 点前累计 {cum[16]:,} 人（{cum[16] / V:.0%}）", (16, cum[16]), xytext=(-8, 14),
            textcoords="offset points", ha="right", fontsize=9.5, color=INK)
ax.annotate(f"19 点累计 {cum[19]:,} 人（{cum[19] / V:.0%}）", (19, cum[19]), xytext=(-8, 12),
            textcoords="offset points", ha="right", fontsize=9.5, color=INK)
ax.text(18, V * 1.02, f"17–19 点新到 {new.loc[17:19].sum():,} 人\n= 到过者的 {p17_19:.0%}", ha="center", va="top",
        fontsize=10, color=MAIN, fontweight="bold", linespacing=1.4)
ax.text(-0.4, 1150, f"0 点已在 {new[0]:,} 人\n（前一晚没走）", va="bottom", fontsize=9, color=MUTED, linespacing=1.3)
ax.set_xlim(-0.6, 23.6); ax.set_ylim(0, V * 1.1)
ax.set_xticks(range(0, 24, 3), [f"{v} 点" for v in range(0, 24, 3)])
ax.yaxis.set_major_formatter("{x:,.0f}")
ax.set_title("西塔 500 米内的人数（人）", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = f"到过西塔的 {V:,} 人里 {p17_19:.0%} 第一次出现在 17–19 点；17 点新到 {new[17]:,} 人，是 16 点的 {jump:.0f} 倍"
viz.finish(fig, title,
           "柱高 = 该小时在西塔 500 米内的人数（0 点的深灰 = 前一晚就在、没走）；浅蓝底 = 17–19 点\n"
           "距离按所连站点坐标计算；口径：dwd_signal × dim_cell",
           SOURCE, "D4_04_西塔在场与首次出现.png")
plt.show()

assert (old >= 0).all() and V == near["imsi"].nunique()
assert pk in (19, 20) and 0.35 < p17_19 < 0.45 and jump > 5.6

In [ ]:
# ── 出图 D4_06：小多图只画和平区周边；左上角全市定位图，放大框连线到第一张 ──
from matplotlib.colors import BoundaryNorm
HOURS = [4, 10, 15, 19, 23]
SHOWN = ["和平区", "沈河区", "皇姑区", "大东区", "铁西区", "于洪区", "浑南区", "沈北新区", "苏家屯区"]
act = uh.groupby("hour").size()                                   # 分母：该小时有信令的人（含市外）
by_d = uh.groupby(["district", "hour"]).size().unstack(fill_value=0).reindex(columns=HOURS, fill_value=0)
pct = by_d.div(act[HOURS], axis=1) * 100
assert (by_d.loc[SHOWN] >= 5).all().all()                         # 框里的区每个时刻都 ≥ 5 人，不用藏
outside = 100 - pct.loc[SHOWN].sum()                              # 框外：辽中、新民、法库、康平和市外
BINS = [0, 1, 2.5, 5, 10, 20, 40, 60]
cmap = viz.maps.heat_cmap().resampled(len(BINS) - 1)
nm = BoundaryNorm(BINS, cmap.N)
AT = {"和平区": (123.405, 41.756), "沈河区": (123.549, 41.814), "大东区": (123.518, 41.866),
      "皇姑区": (123.447, 41.877), "铁西区": (123.335, 41.800), "于洪区": (123.312, 41.884),
      "浑南区": (123.544, 41.729), "沈北新区": (123.455, 41.930), "苏家屯区": (123.300, 41.703)}
ALIAS = {n: n.removesuffix("区").removesuffix("新") for n in SHOWN}
BOX = dict(lng=123.425, lat=41.818, half_m=14_500)

fig = plt.figure(figsize=(12.6, 9.6), layout="constrained")
gs = fig.add_gridspec(2, 3)
ax_city = fig.add_subplot(gs[0, 0])
axes = [fig.add_subplot(gs[i // 3, i % 3]) for i in range(1, 6)]
city.base(ax_city, scale=dict(km=20, x=0.56, y=0.05), north=dict(x=0.92, y=0.97))
city.text(ax_city, dict.fromkeys(["新民市", "辽中区", "法库县", "康平县"], np.nan), fs=8.5, nameprops=dict(color=MUTED))
ax_city.set_title("全市 13 个区县，方框 = 5 张小图的范围", loc="left", fontsize=10.5, color=INK)
for i, (ax, h) in enumerate(zip(axes, HOURS)):
    city.base(ax, scale=False, north=False, fill=False)
    pc = city.fill(ax, pct[h].drop("市外"), norm=nm, cmap=cmap, edgecolor="white", lw=0.9)
    city.edges(ax, ["和平区"], color=INK, lw=1.6)
    viz.maps.zoom(ax_city, ax, BOX["lng"], BOX["lat"], BOX["half_m"], color=MAIN, connect=i == 0)
    if i == 0:
        viz.maps.scale_bar(ax, km=5, x=0.70, y=0.035)
    rt = city.text(ax, pct.loc[SHOWN, h], at=AT, alias=ALIAS, fs=9.5,
                   fmt=lambda v: f"{v:.0f}%" if v >= 10 else f"{v:.1f}%")
    for t in (rt.texts["和平区"], rt.nums["和平区"]):
        t.set(color="white", fontweight="bold", path_effects=[])
    ax.plot(hot_lng, hot_lat, marker="+", ms=9, mew=1.6, color="white", zorder=7)
    ax.set_title(f"{h} 点", loc="left", fontsize=12, color=INK, fontweight="bold")
    ax.set_title(f"有信令 {act[h]:,} 人，框外 {outside[h]:.1f}%", loc="right", fontsize=9.5, color=MUTED)
viz.maps.colorbar(pc, axes, label="该小时在这个区的人，占该小时有信令的人（%，分级色阶）",
                  ticks=BINS, fmt="{:g}%", shrink=0.55)
hp = pct.loc["和平区"]
title = f"23 点 {hp[23]:.0f}% 的人在和平区，10 点只有 {hp[10]:.0f}%；其余城区离西塔越远，夜里留下的越少"
viz.finish(fig, title,
           "颜色 = 该小时位置落在这个区的人数 ÷ 该小时有信令的人数（每人每小时算一次）；五张图同一分级色阶；黑框 = 和平区，+ = 西塔\n"
           "位置 = 该小时信令位置的中位数；框外 = 辽中、新民、法库、康平和市外；口径：dwd_signal × dim_cell",
           SOURCE, "D4_06_各时刻人在哪些区.png")
plt.show()